# Text Embedding — Offline với Sentence Transformers
Không cần API key, chạy hoàn toàn local.
Model sẽ tự download lần đầu (~470MB), sau đó dùng offline.

In [ ]:
# pip install -qU sentence-transformers langchain-huggingface numpy
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# Model đa ngôn ngữ, hỗ trợ tiếng Việt
embeddings = HuggingFaceEmbeddings(
    # model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    # model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    # model_name="BAAI/bge-large-en-v1.5"
)
print("Model loaded.")

/opt/homebrew/Caskroom/miniconda/base/envs/learn_ml/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/learn_ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10488.53it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [8]:
# --- Embed 1 đoạn văn bản ---
text = "Hôm nay trời Hà Nội nhiều mây và mát mẻ."

vector = embeddings.embed_query(text)

print(f"Text    : {text}")
print(f"Số chiều: {len(vector)}")
print(f"10 giá trị đầu: {np.round(vector[:10], 4)}")

Text    : Hôm nay trời Hà Nội nhiều mây và mát mẻ.
Số chiều: 384
10 giá trị đầu: [ 0.2375  0.14    0.0673  0.1772  0.4278  0.1159  0.1421 -0.1958 -0.1144
 -0.0091]


In [4]:
# --- Embed nhiều đoạn văn bản ---
docs = [
    "Hôm nay trời Hà Nội nhiều mây và mát mẻ.",
    "Thời tiết Hà Nội hôm nay có mây, dễ chịu.",
    "Con mèo đang ngủ trên ghế sofa.",
    "Bầu trời Sài Gòn nắng chói chang.",
    "Chiếc xe hơi màu đỏ đậu trước cửa.",
    "Một chiếc ô tô đỏ đang đỗ ở trước nhà.",
]

vectors = embeddings.embed_documents(docs)

print(f"Số văn bản : {len(vectors)}")
print(f"Số chiều   : {len(vectors[0])}")

Số văn bản : 6
Số chiều   : 384


In [5]:
# --- Cosine similarity ---
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = docs[0]  # "Hôm nay trời Hà Nội nhiều mây..."
query_vec = vectors[0]

print(f'Query: "{query}"\n')
for doc, vec in zip(docs, vectors):
    score = cosine_similarity(query_vec, vec)
    print(f"  [{score:.4f}] {doc}")

Query: "Hôm nay trời Hà Nội nhiều mây và mát mẻ."

  [1.0000] Hôm nay trời Hà Nội nhiều mây và mát mẻ.
  [0.8834] Thời tiết Hà Nội hôm nay có mây, dễ chịu.
  [-0.1015] Con mèo đang ngủ trên ghế sofa.
  [0.6333] Bầu trời Sài Gòn nắng chói chang.
  [0.0887] Chiếc xe hơi màu đỏ đậu trước cửa.
  [0.1511] Một chiếc ô tô đỏ đang đỗ ở trước nhà.


In [6]:
# --- Kiểm chứng Semantic: "xe hơi" vs "ô tô" ---
# Hai câu khác từ nhưng cùng nghĩa → vector phải gần nhau
xe_hoi = embeddings.embed_query("chiếc xe hơi màu đỏ")
o_to   = embeddings.embed_query("chiếc ô tô màu đỏ")
meo    = embeddings.embed_query("con mèo đang ngủ")

print("So sánh với 'chiếc xe hơi màu đỏ':")
print(f"  'chiếc ô tô màu đỏ'  → {cosine_similarity(xe_hoi, o_to):.4f}  (cùng nghĩa → gần)")
print(f"  'con mèo đang ngủ'   → {cosine_similarity(xe_hoi, meo):.4f}  (khác nghĩa → xa)")

So sánh với 'chiếc xe hơi màu đỏ':
  'chiếc ô tô màu đỏ'  → 0.9953  (cùng nghĩa → gần)
  'con mèo đang ngủ'   → 0.1106  (khác nghĩa → xa)


In [7]:
# --- Semantic search: tìm văn bản gần nhất ---
def semantic_search(query: str, docs: list[str], doc_vectors: list, top_k: int = 3):
    query_vec = embeddings.embed_query(query)
    scored = [
        (cosine_similarity(query_vec, vec), doc)
        for doc, vec in zip(docs, doc_vectors)
    ]
    scored.sort(reverse=True)
    return scored[:top_k]

query = "trời hôm nay thế nào?"
results = semantic_search(query, docs, vectors)

print(f'Query: "{query}"\n')
for score, doc in results:
    print(f"  [{score:.4f}] {doc}")

Query: "trời hôm nay thế nào?"

  [0.4392] Hôm nay trời Hà Nội nhiều mây và mát mẻ.
  [0.3153] Thời tiết Hà Nội hôm nay có mây, dễ chịu.
  [0.3009] Bầu trời Sài Gòn nắng chói chang.
